# IRWA Project — Part 1
## Text Processing and Exploratory Data Analysis

This notebook contains the implementation and analysis for Part 1 of the IRWA project.

The first stage explores the raw fashion-product corpus and verifies its structure. We then preprocess textual fields while preserving the original product metadata for later retrieval, ranking, filtering, and display.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

Matplotlib is building the font cache; this may take a moment.


In [11]:
PROJECT_ROOT = Path(r"C:\Users\paubo\OneDrive\Desktop\UPF\2026_27\IRWA\IRWA_Project\irwa-search-engine")

DATA_PATH = PROJECT_ROOT / "data" / "fashion_products_dataset.json"
VALIDATION_PATH = PROJECT_ROOT / "data" / "validation_labels.csv"
FIGURES_PATH = PROJECT_ROOT / "project_progress" / "part_1" / "figures"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Validation labels:", VALIDATION_PATH)

Project root: C:\Users\paubo\OneDrive\Desktop\UPF\2026_27\IRWA\IRWA_Project\irwa-search-engine
Dataset: C:\Users\paubo\OneDrive\Desktop\UPF\2026_27\IRWA\IRWA_Project\irwa-search-engine\data\fashion_products_dataset.json
Validation labels: C:\Users\paubo\OneDrive\Desktop\UPF\2026_27\IRWA\IRWA_Project\irwa-search-engine\data\validation_labels.csv


## 1. Dataset loading and initial inspection

In [13]:
df = pd.read_json(DATA_PATH)

print(f"Number of documents: {len(df):,}")
print(f"Number of fields: {len(df.columns)}")

Number of documents: 28,080
Number of fields: 17


In [14]:
df.head(3)

,_id,actual_price,average_rating,brand,category,crawled_at,description,discount,images,out_of_stock,pid,product_details,seller,selling_price,sub_category,title,url
0,fa8e22d6-c0b6-5229-bb9e-ad52eda39a0a,"2,999",3.9,York,Clothing and Accessories,2021-02-10 20:11:51,"Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...",69% off,"[https://rukminim1.flixcart.com/image/128/128/jr3t5e80/track-pant/z/y/n/m-1005combo2-yorker-original-imafczg3xfh5qqd4.jpeg?q=70, https://rukminim1...",False,TKPFCZ9EA7H5FYZH,"[{'Style Code': '1005COMBO2'}, {'Closure': 'Elastic'}, {'Pockets': 'Side Pockets'}, {'Fabric': 'Cotton Blend'}, {'Pattern': 'Solid'}, {'Color': 'M...",Shyam Enterprises,921,Bottomwear,Solid Women Multicolor Track Pants,https://www.flipkart.com/yorker-solid-men-multicolor-track-pants/p/itmd2c76aadce459?pid=TKPFCZ9EA7H5FYZH&lid=LSTTKPFCZ9EA7H5FYZHVYXWP0&marketplace...
1,893e6980-f2a0-531f-b056-34dd63fe912c,"1,499",3.9,York,Clothing and Accessories,2021-02-10 20:11:52,"Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...",66% off,"[https://rukminim1.flixcart.com/image/128/128/kfyasnk0/track-pant/g/5/y/s-19876-yorker-original-imafwamyzrwjynkf.jpeg?q=70, https://rukminim1.flix...",False,TKPFCZ9EJZV2UVRZ,"[{'Style Code': '1005BLUE'}, {'Closure': 'Drawstring, Elastic'}, {'Pockets': 'Side Pockets'}, {'Fabric': 'Cotton Blend'}, {'Pattern': 'Solid'}, {'...",Shyam Enterprises,499,Bottomwear,Solid Men Blue Track Pants,https://www.flipkart.com/yorker-solid-men-blue-track-pants/p/itmfczez7v6rzwer?pid=TKPFCZ9EJZV2UVRZ&lid=LSTTKPFCZ9EJZV2UVRZ9HEITU&marketplace=FLIPK...
2,eb4c8eab-8206-59d0-bcd1-a724d96bf74f,"2,999",3.9,York,Clothing and Accessories,2021-02-10 20:11:52,"Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...",68% off,"[https://rukminim1.flixcart.com/image/128/128/jr3t5e80/track-pant/e/c/s/xl-1005combo4-yorker-original-imafczgcc8t8uppk.jpeg?q=70, https://rukminim...",False,TKPFCZ9EHFCY5Z4Y,"[{'Style Code': '1005COMBO4'}, {'Closure': 'Elastic'}, {'Pockets': 'Side Pockets'}, {'Fabric': 'Cotton Blend'}, {'Pattern': 'Solid'}, {'Color': 'M...",Shyam Enterprises,931,Bottomwear,Solid Men Multicolor Track Pants,https://www.flipkart.com/yorker-solid-men-multicolor-track-pants/p/itm1eee3ca648155?pid=TKPFCZ9EHFCY5Z4Y&lid=LSTTKPFCZ9EHFCY5Z4YDOVNQS&marketplace...


In [15]:
df.columns.tolist()

['_id',
 'actual_price',
 'average_rating',
 'brand',
 'category',
 'crawled_at',
 'description',
 'discount',
 'images',
 'out_of_stock',
 'pid',
 'product_details',
 'seller',
 'selling_price',
 'sub_category',
 'title',
 'url']

In [17]:
required_fields = [
    "pid",
    "title",
    "description",
    "brand",
    "category",
    "sub_category",
    "product_details",
    "seller",
    "out_of_stock",
    "selling_price",
    "discount",
    "actual_price",
    "average_rating",
    "url",
]

missing_required_fields = [
    field for field in required_fields
    if field not in df.columns
]

print("Missing required fields:", missing_required_fields)

assert len(missing_required_fields) == 0, (
    f"Dataset is missing required fields: {missing_required_fields}"
)

Missing required fields: []


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28080 entries, 0 to 28079
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   _id              28080 non-null  object        
 1   actual_price     28080 non-null  object        
 2   average_rating   28080 non-null  object        
 3   brand            28080 non-null  object        
 4   category         28080 non-null  object        
 5   crawled_at       28080 non-null  datetime64[ns]
 6   description      28080 non-null  object        
 7   discount         28080 non-null  object        
 8   images           28080 non-null  object        
 9   out_of_stock     28080 non-null  bool          
 10  pid              28080 non-null  object        
 11  product_details  28080 non-null  object        
 12  seller           28080 non-null  object        
 13  selling_price    28080 non-null  object        
 14  sub_category     28080 non-null  objec

In [20]:
print("Missing PID:", df["pid"].isna().sum())
print("Unique PID:", df["pid"].nunique())
print("Number of rows:", len(df))
print("Duplicated PID:", df["pid"].duplicated().sum())

Missing PID: 0
Unique PID: 28080
Number of rows: 28080
Duplicated PID: 0


In [24]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
})

missing

,missing_count
_id,0
actual_price,0
average_rating,0
brand,0
category,0
crawled_at,0
description,0
discount,0
images,0
out_of_stock,0


In [25]:
sample_columns = [
    "pid",
    "title",
    "description",
    "brand",
    "category",
    "sub_category",
    "product_details",
    "seller",
    "selling_price",
    "discount",
    "actual_price",
    "average_rating",
    "out_of_stock",
]

df[sample_columns].sample(5, random_state=42)

,pid,title,description,brand,category,sub_category,product_details,seller,selling_price,discount,actual_price,average_rating,out_of_stock
22664,SHTFW5YQZGWHZPCJ,Women Regular Fit Washed Mandarin Collar Casual Shirt,"ATM SHIRTS offers quality casual wear in the widest variety which makes it easy to choose, at prices that are easy to reach and excellent service ...",A,Clothing and Accessories,Topwear,"[{'Pack of': '1'}, {'Style Code': '3051_L'}, {'Closure': 'Button'}, {'Fit': 'Regular'}, {'Fabric': 'Denim'}, {'Sleeve': 'Full Sleeve'}, {'Pattern'...",ARYACITION,999,66% off,"2,999",2.8,False
5676,SRTFUY4BJRTVV5J8,Striped Women Dark Blue Sports Shorts,,M7 By Metrona,Clothing and Accessories,Bottomwear,"[{'Fabric': 'Polyester'}, {'Pattern': 'Striped'}, {'Color': 'Dark Blue'}, {'Style Code': 'M7SS20SPSH009B'}, {'Closure': 'Drawstring, Elasticated'}...",RetailNet,485,55% off,"1,099",3.8,False
10356,JCKFZSPYYZGCYGZK,Sleeveless Printed Women Nehru Jacket,,True Bl,Clothing and Accessories,Winter Wear,"[{'Color': 'Black'}, {'Fabric': 'Polyester Blend'}, {'Pattern': 'Printed'}, {'Style Code': '20319122-Black'}, {'Ideal For': 'Women'}, {'Sleeve': '...",KAPSONSRETAILPVTLTD,"1,599",60% off,"3,999",3.6,False
9463,TSHFU7F9HPYDPWMA,Solid Women Round Neck Multicolor T-Shirt,"This casual t-shirt is perfect for everyone like school-college student, working and sports person etc and can be wear anywhere MATERIAL - These a...",FancyW,Clothing and Accessories,Topwear,"[{'Type': 'Round Neck'}, {'Sleeve': 'Half Sleeve'}, {'Fit': 'Regular'}, {'Fabric': 'Cotton Lycra Blend'}, {'Sales Package': '1 T-shirt'}, {'Pack o...",FancyWay,201,77% off,899,4.2,False
428,VESFGYJCZH7GMD4E,SayItLoud Women Vest,The world is boring without a little twist and a twist is what we have added to your favourite Round neck 100% cotton Vest with a quotation '8'. W...,,Clothing and Accessories,Innerwear and Swimwear,"[{'Neck': 'Round Neck'}, {'Pattern': 'Printed'}, {'Sleeve': 'Sleeveless'}, {'Sales Package': 'Single'}, {'Series': 'Latest Vest'}, {'Style Code': ...",sayitloud,299,62% off,799,4.3,False


In [26]:
text_fields = [
    "title",
    "description",
    "brand",
    "category",
    "sub_category",
    "seller",
]

for field in text_fields:
    print(f"\n--- {field.upper()} ---")
    print(df[field].dropna().astype(str).head(10).tolist())


--- TITLE ---
['Solid Women Multicolor Track Pants', 'Solid Men Blue Track Pants', 'Solid Men Multicolor Track Pants', 'Solid Women Multicolor Track Pants', 'Solid Women Brown, Grey Track Pants', 'Solid Women Multicolor Track Pants', 'Solid Women Dark Blue Track Pants', 'Solid Women Beige Track Pants', 'Solid Women Black Track Pants', 'Solid Men Grey Track Pants']

--- DESCRIPTION ---
['Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India', 'Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India', 'Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India', 'Yorker trackpants made from 100% 

In [27]:
for value in df["product_details"].dropna().head(5):
    print(value)
    print(type(value))
    print("-" * 80)

[{'Style Code': '1005COMBO2'}, {'Closure': 'Elastic'}, {'Pockets': 'Side Pockets'}, {'Fabric': 'Cotton Blend'}, {'Pattern': 'Solid'}, {'Color': 'Multicolor'}]
<class 'list'>
--------------------------------------------------------------------------------
[{'Style Code': '1005BLUE'}, {'Closure': 'Drawstring, Elastic'}, {'Pockets': 'Side Pockets'}, {'Fabric': 'Cotton Blend'}, {'Pattern': 'Solid'}, {'Color': 'Blue'}]
<class 'list'>
--------------------------------------------------------------------------------
[{'Style Code': '1005COMBO4'}, {'Closure': 'Elastic'}, {'Pockets': 'Side Pockets'}, {'Fabric': 'Cotton Blend'}, {'Pattern': 'Solid'}, {'Color': 'Multicolor'}]
<class 'list'>
--------------------------------------------------------------------------------
[{'Style Code': '1005COMBO3'}, {'Closure': 'Elastic'}, {'Pockets': 'Side Pockets'}, {'Fabric': 'Cotton Blend'}, {'Pattern': 'Solid'}, {'Color': 'Multicolor'}]
<class 'list'>
---------------------------------------------------------

In [28]:
from preprocessing import (
    normalize_text,
    tokenize_text,
    remove_stopwords,
    stem_tokens,
    preprocess_text,
)

example = "Women's Full-Sleeve Cotton Sweatshirts for the Winter!"

normalized = normalize_text(example)
tokens = tokenize_text(example)
without_stopwords = remove_stopwords(tokens)
stemmed = stem_tokens(without_stopwords)

print("Original:")
print(example)

print("\nNormalized:")
print(normalized)

print("\nTokens:")
print(tokens)

print("\nWithout stop words:")
print(without_stopwords)

print("\nStemmed:")
print(stemmed)

Original:
Women's Full-Sleeve Cotton Sweatshirts for the Winter!

Normalized:
women full sleeve cotton sweatshirts for the winter!

Tokens:
['women', 'full', 'sleeve', 'cotton', 'sweatshirts', 'for', 'the', 'winter']

Without stop words:
['women', 'full', 'sleeve', 'cotton', 'sweatshirts', 'winter']

Stemmed:
['women', 'full', 'sleev', 'cotton', 'sweatshirt', 'winter']


In [29]:
df["title_tokens"] = df["title"].apply(preprocess_text)
df["description_tokens"] = df["description"].apply(preprocess_text)

In [30]:
df[
    [
        "pid",
        "title",
        "title_tokens",
        "description",
        "description_tokens",
    ]
].head(10)

,pid,title,title_tokens,description,description_tokens
0,TKPFCZ9EA7H5FYZH,Solid Women Multicolor Track Pants,"[solid, women, multicolor, track, pant]","Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...","[yorker, trackpant, made, 100, rich, comb, cotton, give, rich, look, design, comfort, skin, friend, fabric, itch, free, waistband, great, year, ro..."
1,TKPFCZ9EJZV2UVRZ,Solid Men Blue Track Pants,"[solid, men, blue, track, pant]","Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...","[yorker, trackpant, made, 100, rich, comb, cotton, give, rich, look, design, comfort, skin, friend, fabric, itch, free, waistband, great, year, ro..."
2,TKPFCZ9EHFCY5Z4Y,Solid Men Multicolor Track Pants,"[solid, men, multicolor, track, pant]","Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...","[yorker, trackpant, made, 100, rich, comb, cotton, give, rich, look, design, comfort, skin, friend, fabric, itch, free, waistband, great, year, ro..."
3,TKPFCZ9ESZZ7YWEF,Solid Women Multicolor Track Pants,"[solid, women, multicolor, track, pant]","Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...","[yorker, trackpant, made, 100, rich, comb, cotton, give, rich, look, design, comfort, skin, friend, fabric, itch, free, waistband, great, year, ro..."
4,TKPFCZ9EVXKBSUD7,"Solid Women Brown, Grey Track Pants","[solid, women, brown, grey, track, pant]","Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...","[yorker, trackpant, made, 100, rich, comb, cotton, give, rich, look, design, comfort, skin, friend, fabric, itch, free, waistband, great, year, ro..."
5,TKPFCZ9EFK9DNWDA,Solid Women Multicolor Track Pants,"[solid, women, multicolor, track, pant]","Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...","[yorker, trackpant, made, 100, rich, comb, cotton, give, rich, look, design, comfort, skin, friend, fabric, itch, free, waistband, great, year, ro..."
6,TKPFDABN3GXYPFHE,Solid Women Dark Blue Track Pants,"[solid, women, dark, blue, track, pant]","Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...","[yorker, trackpant, made, 100, rich, comb, cotton, give, rich, look, design, comfort, skin, friend, fabric, itch, free, waistband, great, year, ro..."
7,TKPFCZ9ESGZYT8NH,Solid Women Beige Track Pants,"[solid, women, beig, track, pant]","Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...","[yorker, trackpant, made, 100, rich, comb, cotton, give, rich, look, design, comfort, skin, friend, fabric, itch, free, waistband, great, year, ro..."
8,TKPFCZ9DYU33FFXS,Solid Women Black Track Pants,"[solid, women, black, track, pant]","Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...","[yorker, trackpant, made, 100, rich, comb, cotton, give, rich, look, design, comfort, skin, friend, fabric, itch, free, waistband, great, year, ro..."
9,TKPFDABN4NQFVKZY,Solid Men Grey Track Pants,"[solid, men, grey, track, pant]","Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great fo...","[yorker, trackpant, made, 100, rich, comb, cotton, give, rich, look, design, comfort, skin, friend, fabric, itch, free, waistband, great, year, ro..."

In [31]:
assert len(df) == len(df["title_tokens"])
assert len(df) == len(df["description_tokens"])

assert df["title_tokens"].apply(lambda x: isinstance(x, list)).all()
assert df["description_tokens"].apply(lambda x: isinstance(x, list)).all()

print("All title and description fields processed successfully.")

All title and description fields processed successfully.
